# p10 Student-t · Spline-Ridge Mean-centered Huber-DML 全量实验

n=1000，p=10，R=1000，DGP1/DGP2，Gaussian 和 t(20,10,5,3,2.6,2.3,2.1)，共 16,000 组数据、64,000 条方法记录。新增协变量保持 AR(1) 相关性但真实系数为零，真实函数只使用前五列。四种方法为原 M1、M2、M3 和 S，S 使用算术均值位置 A 和 adjusted Score-test。

使用 Python (environment2) 内核，顺序执行。MODE 支持 smoke/full，full 必须先自检。默认执行本次授权的 full。所有结果和断点独立保存，重复执行会续跑。RETRY_FAILED=True 显式重试失败方法，历史失败记录仍保留。

## 必要修改与核验

P 改为 10，生成器使用实际 p，真实函数仅提取前五列。所有估计器、Spline-Ridge 特征及超参数、二阶段 Huber 和 Dukes corrected score 保持不变。Smoke 逐函数核验源码，验证 8 个 Worker、全 10 列输入、115 个样条及交互展开特征、串并行一致性、断点恢复、内外折隔离、p5 原 S 函数在同一十维数据上的复算，以及点估计的真值隔离。

不使用历史 p5 数据复放，不以 Smoke Coverage 接近 95% 作为通过条件。M3 增加的净交叉项只是事后诊断，保持原 transformed-outcome Y MSE 的目标。p10 与 p5 只在场景级比较。

In [ ]:
from pathlib import Path
import sys
import multiprocessing
from IPython.display import Image, Markdown, display

HERE = Path.cwd()
RESULTS = HERE / "Spline_Ridge_DML_p10_StudentT_Full"
if not (RESULTS / "spline_p10_full_worker.py").exists():
    HERE = Path.home() / "Desktop" / "毕设DML实验" / "DML_SplineRidge p10 Student-t全量实验"
    RESULTS = HERE / "Spline_Ridge_DML_p10_StudentT_Full"
sys.path.insert(0, str(RESULTS))
from spline_p10_full_worker import config_for, smoke_run, full_run
MODE = "full"
RETRY_FAILED = False
assert MODE in ("smoke", "full")
settings = config_for(MODE)
print(f"p={settings['p']}，n={settings['n']}，R={settings['R']}，8 Worker × 1 数值线程")


## Smoke 自检

8 组数据、32 条方法记录单独保存，完成内部确定性和断点校验后才运行 Full。

In [ ]:
if __name__ == "__main__":
    multiprocessing.freeze_support()
    smoke = smoke_run(RESULTS / "smoke")
    print(f"Smoke 通过，p={smoke['p']}，{smoke['data_n']} 组数据，"
          f"{smoke['point_n']} 个点估计和 {smoke['test_n']} 个检验有效。")


## 正式 Full R1000

Windows spawn，8 个 Worker，每个 Worker 数值线程为 1；每 32 组提交 SQLite 事务。数据、fold 和 seed 与执行顺序无关。无 Lognormal 场景。原始记录、失败和重试历史保留。

In [ ]:
if __name__ == "__main__":
    if not smoke["passed"]:
        raise RuntimeError("Smoke 未通过")
    results = full_run(RESULTS, retry_failed=RETRY_FAILED) if MODE == "full" else None
    if results is not None:
        print(f"完成 {results['data_n']:,} 组数据；有效点估计 {results['point_n']:,}，检验 {results['test_n']:,}。")


## 完整汇总、配对比较和图表

In [ ]:
if results is not None:
    display(results["summary"].round(6))
    display(results["paired"].round(6))
    for filename in ("coverage_student_t.png", "deterioration_student_t.png",
                     "bias_rmse_student_t.png", "score_diagnostics.png",
                     "nuisance_diagnostics.png", "p10_vs_p5.png"):
        display(Image(filename=str(RESULTS / filename), width=1100))


## 结果解释与跨维度场景级比较

In [ ]:
if results is not None:
    display(Markdown(results["findings"]))
